In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import html
import time, json
from pathlib import Path
from datasets import load_dataset

MODEL_REPO = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MODEL_REVISION = "f15c379fb32bb402fa06a7ae9aecb1febf4b79ec"


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")

ROOT = find_repo_root()
SPLITS_DIR = ROOT / "splits"

hotpot_ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

In [2]:
PROMPT_VERSION = "v1"
MAX_NEW_TOKENS = 32

SYSTEM_WITH_CONTEXT ="""
You answe questions using the paragraph provided. Reply with only the answer: a short phrase, copied from the paragraphs where possible, with no explanation and no full sentence.
If the question is a yes/no question, reply exactly with exactly yes or no
"""

SYSTEM_CLOSED_BOOK = """ 
You answer questions from your own knowledge. 
Reply with only the answer: a short phrase, with no explanation and no full sentence. If the question is a yes/no question, reply with exactly yes or no.
"""

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_REPO, revision=MODEL_REVISION, device_map="cuda")
model.eval()
model.generation_config.max_length = None

print(torch.cuda.memory_allocated() / 1e9)
print(model.config.quantization_config)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

5.702755328
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": [
    "lm_head",
    "multi_modal_projector",
    "merger",
    "modality_projection"
  ],
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [4]:
def build_messages(question, paragraphs=None):
    """Chat messages for one question. paragraphs: [{"title", "text"}, ...] in rank order, or None for closed-book."""
    if not paragraphs:
        return [
            {"role": "system", "content": SYSTEM_CLOSED_BOOK},
            {"role": "user", "content": f"Question: {question}"}
        ]
    context = "\n\n".join(
        f"[{i}] {html.unescape(p['title'])}\n{p['text']}" for i, p in enumerate(paragraphs, start=1)
    )

    return [
        {"role": "system", "content": SYSTEM_WITH_CONTEXT},
        {"role": "user", "content": f"Paragraphs: \n\n{context}\n\nQuestion: {question}"}
    ]


def encode_prompt(question, paragraphs=None):
    """Chat prompt as model inputs (input_ids + attention_mask), ending where the answer starts."""
    return tokenizer.apply_chat_template(
        build_messages(question, paragraphs),
        add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

@torch.no_grad()
def generate_raw(question, paragraphs=None, max_new_tokens=MAX_NEW_TOKENS):
    """Greedy answer text before any post-processing (only the newly generated tokens)."""
    inputs = encode_prompt(question, paragraphs)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    
    return tokenizer.decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)

In [5]:
with open(SPLITS_DIR / "pilot.json") as f:
    pilot_ids = json.load(f)
            
pilot_q = (hotpot_ds["train"]
            .select_columns(["id", "question", "answer", "type", "context"])
            .to_pandas().set_index("id").loc[pilot_ids])
    
def distractor_paragraphs(context):
    """The 10 paragraphs given with the question (2 gold + 8 distractors), in their given order."""
    return [{"title": title, "text": "".join(sentences)}
            for title, sentences in zip(context["title"], context["sentences"])]
    
for qid, row in pilot_q.head(3).iterrows():
    paragraphs = distractor_paragraphs(row.context)
    start = time.time()
    raw = generate_raw(row.question, paragraphs)
    print(f"[{row.type}] {row.question}")
    print(f"  gold : {row.answer!r}")
    print(f"  raw  : {raw!r}")
    print(f"  {len(paragraphs)} paragraphs, {time.time() - start:.1f} s\n")

[bridge] Susan Barnes cofound NeXT Compiter, Inc. with an entrepreneur born in which year ?
  gold : 'Steve Jobs'
  raw  : '1985'
  10 paragraphs, 0.8 s

[bridge] Taya Kyle was the widow of this US Navy SEAL who served how many tours in the Iraq War?
  gold : 'four'
  raw  : 'four tours.'
  10 paragraphs, 0.9 s

[bridge] What is the theme of the council established by Bill Clinton that preserved the work of Joann Wheeler Kealiinohomoku under the "Save America's Treasures" program?
  gold : 'Honor the Past – Imagine the Future'
  raw  : '"Honor the Past – Imagine the Future."'
  10 paragraphs, 1.2 s

